# preperations

In [1]:
import numpy as np
import pandas as pd

In [2]:
train_original = pd.read_csv("data/train.csv")
test_original = pd.read_csv("data/test.csv")

In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [4]:
train_original.head()

,kaltmiete_listing,overpriced,floor,furnished,num_photos,total_floors,building_age_years,listing_date,wg_typ,year_built,priority_score,num_rooms,plz,balcony,days_online,living_area_m2,stadtteil,living_area_text,energieausweis,dist_uni_km,contact_type,heating_type,kitchen,dist_uni_min,elevator,clicks_last_week,listing_id,nebenkosten_eur
0,325 €,False,2. OG,teilmöbliert,16,2,126,08.05.2026,gemischt,1900,7,3.0,93051,nein,52,21.2,Königswiesen,21.2 m2,A+,7.5,Genossenschaft,Nachtspeicher,Pantry,30,ja,211,INS-2026-49534,57
1,335 €,True,3. OG,möbliert,10,2,105,04.04.2026,gemischt,1921,9,2.0,93055,ja,71,18.8,Burgweinting,18.8 m2,A+,10.8,Makler,Gasetagenheizung,Einbauküche,44,nein,52,INS-2026-39524,126
2,491 warm,False,EG,unmöbliert,10,2,50,18.03.2026,Studenten-WG,1976,5,1.0,93053,ja,40,16.3,Galgenberg,16.3 m2,E,10.7,Privat,Gasetagenheizung,Einbauküche,43,nein,65,INS-2026-45414,157
3,373 €,False,2. OG,unmöbliert,10,2,18,19.04.2026,Studenten-WG,2008,6,3.0,93059,nein,23,20.4,Stadtamhof,"20,4 qm",E,10.9,Privat,Fernwärme,Einbauküche,44,nein,227,INS-2026-36290,164
4,328,False,EG,teilmöbliert,10,4,35,16.02.2026,Studenten-WG,1991,7,3.0,93049,nein,71,22.5,Karthaus-Prüll,22 m²,D,9.4,Privat,Zentralheizung,Einbauküche,37,nein,94,INS-2026-68392,149


In [5]:
print(train_original.columns)
print(test_original.columns)
print(train_original.dtypes)
print(test_original.dtypes)

Index(['kaltmiete_listing', 'overpriced', 'floor', 'furnished', 'num_photos',
       'total_floors', 'building_age_years', 'listing_date', 'wg_typ',
       'year_built', 'priority_score', 'num_rooms', 'plz', 'balcony',
       'days_online', 'living_area_m2', 'stadtteil', 'living_area_text',
       'energieausweis', 'dist_uni_km', 'contact_type', 'heating_type',
       'kitchen', 'dist_uni_min', 'elevator', 'clicks_last_week', 'listing_id',
       'nebenkosten_eur'],
      dtype='object')
Index(['floor', 'furnished', 'num_photos', 'total_floors',
       'building_age_years', 'listing_date', 'wg_typ', 'year_built',
       'priority_score', 'num_rooms', 'plz', 'balcony', 'days_online',
       'living_area_m2', 'stadtteil', 'living_area_text', 'energieausweis',
       'dist_uni_km', 'contact_type', 'heating_type', 'kitchen',
       'dist_uni_min', 'elevator', 'clicks_last_week', 'listing_id',
       'nebenkosten_eur'],
      dtype='object')
kaltmiete_listing      object
overpriced         

# data cleaning and preprocessing

### orga

- DONE kaltmiete_listing - muss preprocessed werden
- DONE overpriced ist sauber
- DONE floor kann gemapped werden (von -1 bis 4) - weitere spalte is_dachgeschoss[True,False] kann hinzugefügt werden zum trainieren
- DONE furnished keine Ahnung ob man das ordinal machen sollte (habs mal ordinal gemacht)
- DONE num_photos droppen
- DONE total_floors wird behalten
- DONE building_age_years wird behalten, ist besser als year_built
- DONE listing_date droppen
- DONE wg_typ wird behalten, denke nicht dass man das ordinal machen sollte. Habs nicht direkt ordinal, aber mit scikit preprocessor transformiert
- DONE year_built droppen
- POTENTIAL_DROP priority_score - !!! Keine Ahnung was das ist!!! kann vermutlich gedroppt werden, wird final mit und ohne getestet
- DONE num_rooms wird behalten
- POTENTIAL_DROP plz wird in String umgewandelt da es keine sinnvolle Ordnung als Zahl dafür gibt
- DONE balcony wird in True/False umgewandelt
- DONE days_online droppen
- DONE living_area_m2 wird behalten, das ist sauber (wird mit benutzt um outlyer in den preis/qm zu droppen)
- POTENTIAL_DROP stadtteil wird behalten - evtl mit plz blabla doppelter einfluß
- DONE living_area_text droppen, sind großteils bei living_area_m2, aber weichen manchmal ab
- DONE energieausweis wird ordinal gemacht
- POTENTIAL_DROP dist_uni_km wird behalten
- DONE contact_type wird behalten
- DONE heating_Type wird behalten
- DONE kitchen wird behalten, ordinal machen könnte sinn machen aber nicht implementiert. Ich hätte es nicht ordinal gemacht.
- POTENTIAL_DROP dist_uni_min wird behalten, ist ein bissl was anderes als dist_uni_km inmyopinion - aber vllt könnte man eins von beiden droppen
- DONE elevator wird in True/False umgewandelt
- DONE clicks_last_week wird behalten - könnte vorraussagen ob die miete für die qm preise attraktiv ist
- DONE listing_id droppen
- DONE nebenkosten_eur wird behalten

In [6]:


for col in train_original.columns:
    print(col)
    print(train_original[col].unique())
    print("-" * 40)


kaltmiete_listing
['325 €' '335 €' '491 warm' ... '290' '530' 'Kaltmiete: 305 €']
----------------------------------------
overpriced
[False  True]
----------------------------------------
floor
['2. OG' '3. OG' 'EG' 'Souterrain' 'DG' '4. OG' '1. OG']
----------------------------------------
furnished
['teilmöbliert' 'möbliert' 'unmöbliert']
----------------------------------------
num_photos
[16 10 21 14 15  6  8  7 19 18  5 12  2  4 23  9 11 25  3 13 24 17 20 22]
----------------------------------------
total_floors
[2 4 1 3 7 6 5]
----------------------------------------
building_age_years
[126 105  50  18  35  15  84   4  30 101 123  92  52 104  76  57  85  47
  78 112  26   9 109  63 122  12  71   2  68  77  91  25   8  17  21 102
 119  51  61  72  33  48  22 113  98  41  42  29  64  60 115  74  45  90
 106  23  28  14  73  24  43  81  87  44  16 116  31  58 100  67  20 121
  46  70  97  10  86  65  93 107 124  38  40  34 110  59  75  66   7  32
 114  99 125  83  80  95  19  96  6

### define cleaning functions

In [7]:
def drop_unleidige_columns(df: pd.DataFrame, cols_to_be_dropped: list) -> pd.DataFrame:
    # print("entferne unleidige Spalten")
    df=df.drop(cols_to_be_dropped, axis=1)
    return df


def mapping_floor_and_add_col_is_dachgeschoss(df: pd.DataFrame) -> pd.DataFrame:
    # print("schreibe floor in ordinal um und füge spalte is_dachgeschoss hinzu")
    df["is_dachgeschoss"] = df["floor"] == "DG"
   
    df["floor_num"] = df["floor"].map({
                                        "Souterrain": -1,
                                        "EG": 0,
                                        "1. OG": 1,
                                        "2. OG": 2,
                                        "3. OG": 3,
                                        "4. OG": 4
                                    })
    #DG ist special
    df.loc[df["floor"] == "DG", "floor_num"] = df["total_floors"]
    df["floor_num"] = df["floor_num"].astype(int)

    df = df.drop(["floor"], axis=1)
    return df

    
def clean_kaltmiete_listing(df: pd.DataFrame) -> pd.DataFrame:
    # print("Bereinige Kaltmiete_listing")
    df['kaltmiete_listing'] = (
        df['kaltmiete_listing'].str.replace('€', '', regex=False)
                                .str.replace('EUR', '', regex=False)
                                .str.replace(',00', '', regex=False)
                                .str.replace('kalt', '', regex=False)
                                .str.replace('Kaltmiete:', '', regex=False)
                                .str.replace(' ', '', regex=False)
                                .str.strip()
                              )
    #nebenkosten rausrechnen
    df["ist_warm"] = df["kaltmiete_listing"].str.contains("warm", na=False) #nur zur kontrolle
    df["kaltmiete_vorher"] = df["kaltmiete_listing"] #nur zur kontrolle
    mask = df["kaltmiete_listing"].str.contains("warm", na=False)
    df.loc[mask, "kaltmiete_listing"] = (
        df.loc[mask, "kaltmiete_listing"]
            .str.replace("warm", "", regex=False)
            .astype(int)
            - df.loc[mask, "nebenkosten_eur"]
    )

    #in Zahl umwandeln
    df["kaltmiete_listing"] = (
        pd.to_numeric(df["kaltmiete_listing"], errors="coerce")
            .fillna(0)
            .astype(int)
    )
    return df


def qm_preis_berechnen(df: pd.DataFrame) -> pd.DataFrame:
    # print("Berechnet qm Preise")
    df["kaltmiete_qm"] = df["kaltmiete_listing"] / df["living_area_m2"]
    return df


def drop_absurde_qm_preise(df:pd.DataFrame, min=0, max=100) -> pd.DataFrame:
    # print(f"Entferne preis/qm outlier, mindestpreis= {min}, höchstpreis={max}")
    # print(f"{len(df)} Einträge im df")
    # print(f"droppe {len(df[df["kaltmiete_qm"] <= min])} Einträge weil qm preis <={min}")
    # print(f"droppe {len(df[df["kaltmiete_qm"] >= max])} Einträge weil qm preis >={max}")
    df = df[df["kaltmiete_qm"] >= min]
    df = df[df["kaltmiete_qm"] <= max]
    # print(f"{len(df)} Einträge im df")
    return df


def mapping(df:pd.DataFrame) -> pd.DataFrame:
    # print("mappe balcony, energieausweis,elevator")
    df["balcony"] = df["balcony"].map({"ja": True, "nein": False})
    df["energieausweis"] = df["energieausweis"].map({"A+": 8,
                                                    "A": 7,
                                                    "B": 6,
                                                    "C": 5,
                                                    "D": 4,
                                                    "E": 3,
                                                    "F": 2,
                                                    "G": 1,
                                                    "H": 0})
    df["elevator"] = df["elevator"].map({"ja": True, "nein": False})
    df["furnished"] = df["furnished"].map({"unmöbliert": 0, "teilmöbliert": 1, "möbliert": 2})
    return df


def plz_als_str(df:pd.DataFrame) -> pd.DataFrame:
    # print("plz als string")
    df["plz"] = df["plz"].astype(str) #PLZ sind keine echten Zahlen, sondern IDs für Regionen.als Kategorie encoden
    return df


def drop_helfer_spalten(df:pd.DataFrame):
    # print("droppe helferspalten")
    df = df.drop(["kaltmiete_vorher", "ist_warm", "kaltmiete_qm"], axis=1)
    return df


def add_room_size_m2(df:pd.DataFrame) -> pd.DataFrame:
    df["room_size_m2"] = df["living_area_m2"] / df["num_rooms"].replace(0, 1)
    return df


def add_luxury_score(df:pd.DataFrame) -> pd.DataFrame:
    df["luxury_score"] = (
        df["balcony"].astype(int) + 
        df["elevator"].astype(int) + 
        (df["kitchen"] == "Einbauküche").astype(int)
    )
    return df


def add_high_floor_without_elevator_penalty(df:pd.DataFrame) -> pd.DataFrame:
    df["high_floor_no_elevator"] = ((df["floor_num"] >= 3) & (~df["elevator"])).astype(int)
    return df
    

def add_student_uni_penalty(df:pd.DataFrame) -> pd.DataFrame:
    df["student_uni_penalty"] = (df["wg_typ"] == "Studenten-WG").astype(int) * df["dist_uni_min"]
    return df


In [8]:
def clean_me_pd(df:pd.DataFrame, is_train=False) -> pd.DataFrame:
    df = mapping_floor_and_add_col_is_dachgeschoss(df)
    if is_train:
        df = clean_kaltmiete_listing(df)
        df = qm_preis_berechnen(df)
        df = drop_absurde_qm_preise(df,8,35)
        df = drop_helfer_spalten(df)
    df = mapping(df)
    df = plz_als_str(df)
    df = add_room_size_m2(df)
    df = add_luxury_score(df)
    df = add_high_floor_without_elevator_penalty(df)
    df = add_student_uni_penalty(df)
    df = drop_unleidige_columns(df, ["listing_id", "living_area_text","days_online","year_built","listing_date", "num_photos", "priority_score","clicks_last_week","nebenkosten_eur", "balcony"])

    return df

In [9]:
pd_cleaned_data = clean_me_pd(train_original.copy(), is_train=True)
display(train_original.head(10))
display(pd_cleaned_data.head(10))

,kaltmiete_listing,overpriced,floor,furnished,num_photos,total_floors,building_age_years,listing_date,wg_typ,year_built,priority_score,num_rooms,plz,balcony,days_online,living_area_m2,stadtteil,living_area_text,energieausweis,dist_uni_km,contact_type,heating_type,kitchen,dist_uni_min,elevator,clicks_last_week,listing_id,nebenkosten_eur
0,325 €,False,2. OG,teilmöbliert,16,2,126,08.05.2026,gemischt,1900,7,3.0,93051,nein,52,21.2,Königswiesen,21.2 m2,A+,7.5,Genossenschaft,Nachtspeicher,Pantry,30,ja,211,INS-2026-49534,57
1,335 €,True,3. OG,möbliert,10,2,105,04.04.2026,gemischt,1921,9,2.0,93055,ja,71,18.8,Burgweinting,18.8 m2,A+,10.8,Makler,Gasetagenheizung,Einbauküche,44,nein,52,INS-2026-39524,126
2,491 warm,False,EG,unmöbliert,10,2,50,18.03.2026,Studenten-WG,1976,5,1.0,93053,ja,40,16.3,Galgenberg,16.3 m2,E,10.7,Privat,Gasetagenheizung,Einbauküche,43,nein,65,INS-2026-45414,157
3,373 €,False,2. OG,unmöbliert,10,2,18,19.04.2026,Studenten-WG,2008,6,3.0,93059,nein,23,20.4,Stadtamhof,"20,4 qm",E,10.9,Privat,Fernwärme,Einbauküche,44,nein,227,INS-2026-36290,164
4,328,False,EG,teilmöbliert,10,4,35,16.02.2026,Studenten-WG,1991,7,3.0,93049,nein,71,22.5,Karthaus-Prüll,22 m²,D,9.4,Privat,Zentralheizung,Einbauküche,37,nein,94,INS-2026-68392,149
5,"301,00 EUR",True,Souterrain,teilmöbliert,21,1,15,22.05.2026,Frauen-WG,2011,4,2.0,93051,ja,51,20.6,Dechbetten,20.6 m2,A+,1.8,Privat,Nachtspeicher,Pantry,7,ja,114,INS-2026-90560,83
6,339,True,DG,unmöbliert,14,2,84,04.01.2026,Frauen-WG,1942,8,2.0,93059,ja,7,17.7,Reinhausen,18 m²,A+,3.4,Makler,Öl,Einbauküche,14,ja,125,INS-2026-87234,74
7,467 warm,False,DG,unmöbliert,15,2,4,13.02.2026,gemischt,2022,8,3.0,93053,nein,52,21.7,Kasernenviertel,"21,7 qm",D,4.3,Privat,Gasetagenheizung,Einbauküche,17,nein,220,INS-2026-90121,85
8,345 warm,False,DG,unmöbliert,6,2,30,17.02.2026,Männer-WG,1996,10,3.0,93055,nein,61,15.1,Burgweinting,15 m²,D,6.2,Privat,Gasetagenheizung,Pantry,24,ja,185,INS-2026-70439,132
9,306 €,False,2. OG,unmöbliert,8,4,101,22.04.2026,Frauen-WG,1925,7,1.0,93049,nein,18,17.7,Westenviertel,18 m²,A,8.6,Privat,Gasetagenheizung,keine,35,nein,297,INS-2026-92509,103


,kaltmiete_listing,overpriced,furnished,total_floors,building_age_years,wg_typ,num_rooms,plz,living_area_m2,stadtteil,energieausweis,dist_uni_km,contact_type,heating_type,kitchen,dist_uni_min,elevator,is_dachgeschoss,floor_num,room_size_m2,luxury_score,high_floor_no_elevator,student_uni_penalty
0,325,False,1,2,126,gemischt,3.0,93051,21.2,Königswiesen,8,7.5,Genossenschaft,Nachtspeicher,Pantry,30,True,False,2,7.066667,1,0,0
1,335,True,2,2,105,gemischt,2.0,93055,18.8,Burgweinting,8,10.8,Makler,Gasetagenheizung,Einbauküche,44,False,False,3,9.400000,2,1,0
2,334,False,0,2,50,Studenten-WG,1.0,93053,16.3,Galgenberg,3,10.7,Privat,Gasetagenheizung,Einbauküche,43,False,False,0,16.300000,2,0,43
3,373,False,0,2,18,Studenten-WG,3.0,93059,20.4,Stadtamhof,3,10.9,Privat,Fernwärme,Einbauküche,44,False,False,2,6.800000,1,0,44
4,328,False,1,4,35,Studenten-WG,3.0,93049,22.5,Karthaus-Prüll,4,9.4,Privat,Zentralheizung,Einbauküche,37,False,False,0,7.500000,1,0,37
5,301,True,1,1,15,Frauen-WG,2.0,93051,20.6,Dechbetten,8,1.8,Privat,Nachtspeicher,Pantry,7,True,False,-1,10.300000,2,0,0
6,339,True,0,2,84,Frauen-WG,2.0,93059,17.7,Reinhausen,8,3.4,Makler,Öl,Einbauküche,14,True,True,2,8.850000,3,0,0
7,382,False,0,2,4,gemischt,3.0,93053,21.7,Kasernenviertel,4,4.3,Privat,Gasetagenheizung,Einbauküche,17,False,True,2,7.233333,1,0,0
8,213,False,0,2,30,Männer-WG,3.0,93055,15.1,Burgweinting,4,6.2,Privat,Gasetagenheizung,Pantry,24,True,True,2,5.033333,1,0,0
9,306,False,0,4,101,Frauen-WG,1.0,93049,17.7,Westenviertel,7,8.6,Privat,Gasetagenheizung,keine,35,False,False,2,17.700000,0,0,0


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def get_scikit_preprocessor():
    categorical_features = ["wg_typ", "plz", "stadtteil", "contact_type", "heating_type", "kitchen"]
    # Alle Spalten, die kontinuierliche Zahlenwerte oder ordinale Werte (wie floor_num) enthalten
    numeric_features = [
        "total_floors", "building_age_years", "num_rooms", 
        "living_area_m2", "dist_uni_km", "dist_uni_min", 
        "floor_num", "furnished", "room_size_m2",
        "luxury_score", "high_floor_no_elevator", "student_uni_penalty"
    ]
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features)
        ],
        remainder="passthrough" # Lässt Booleans wie 'balcony' und 'elevator' unverändert durch
    )
    return preprocessor

# machine learning approach

In [11]:
train = train_original.copy()
train = clean_me_pd(train, is_train=True)
preprocessor = get_scikit_preprocessor()

In [12]:
X = train.drop(["kaltmiete_listing", "overpriced"], axis=1)
y_reg = train["kaltmiete_listing"]
y_class = train["overpriced"]

### regression

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

In [14]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y_reg, test_size=0.2, random_state=42)

#### find optimal hyperparams with RandomSearchGV

In [15]:
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(random_state=42, n_jobs=-1))
])

param_dist_xgb = {
    'model__n_estimators': randint(100, 800),
    'model__learning_rate': uniform(0.01, 0.29), # 0.01 bis 0.3
    'model__max_depth': randint(3, 10),
    'model__subsample': uniform(0.5, 0.5),       # 0.5 bis 1.0 (wie viele Datenpunkte pro Baum)
    'model__colsample_bytree': uniform(0.5, 0.5) # 0.5 bis 1.0 (wie viele Features pro Baum)
}

random_search_xgb = RandomizedSearchCV(
    xgb_pipeline, 
    param_distributions=param_dist_xgb, 
    n_iter=50, 
    cv=10, 
    scoring='r2', 
    n_jobs=-1,
    random_state=42
)

random_search_xgb.fit(X_train, y_train)

y_pred_xgb = random_search_xgb.best_estimator_.predict(X_valid)
score_xgb = r2_score(y_valid, y_pred_xgb)

print(f"Beste Parameter XGBoost: {random_search_xgb.best_params_}")
print(f"R2 Score: {score_xgb:.4f}") # 0.9311

Beste Parameter XGBoost: {'model__colsample_bytree': 0.5165253664502742, 'model__learning_rate': 0.11007066192773805, 'model__max_depth': 3, 'model__n_estimators': 502, 'model__subsample': 0.8403527257773834}
R2 Score: 0.9311


#### optimal hyperparams

In [16]:
XGB_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(colsample_bytree=0.5165253664502742, learning_rate=0.11007066192773805, max_depth=3, n_estimators=502, subsample=0.8403527257773834))
])


X_train, X_valid, y_train, y_valid = train_test_split(X, y_reg, test_size=0.2, random_state=42)

XGB_reg_pipeline.fit(X_train, y_train)

y_pred = XGB_reg_pipeline.predict(X_valid)

score = r2_score(y_valid, y_pred)
print(f"R2 Score: {score:.4f}") # 0.9332

R2 Score: 0.9332


#### predict test data

In [17]:
def check_file_regression(file_path):
    """Check that a regression-prediction CSV is valid for submission to Task 1."""
    try:
        predictions = pd.read_csv(file_path)
    except FileNotFoundError:
        print("File not found or could not be loaded.")
        return False
    if predictions.shape[0] != 500:
        print(f"Expected 500 rows, got {predictions.shape[0]}.")
        return False
    if predictions.shape[1] != 1:
        print("The file should contain exactly one column.")
        return False
    if predictions.columns[0] != "kaltmiete":
        print("The column name should be 'kaltmiete'.")
        return False
    if not np.issubdtype(predictions["kaltmiete"].dtype, np.number):
        print("The values in the 'kaltmiete' column should be numeric.")
        return False
    print("The file is a valid submission for Task 1.")
    return True

In [18]:
test = test_original.copy()

X_test = clean_me_pd(test, is_train=False)

XGB_reg_pipeline.fit(X, y_reg) # Noch einmal auf allen Daten trainieren für maximale Power
final_predictions = XGB_reg_pipeline.predict(X_test)

submission_df = pd.DataFrame({
    "kaltmiete": final_predictions
})

submission_filename = "kaltmiete_prediction.csv"
submission_df.to_csv(submission_filename, index=False)
print(f"Datei '{submission_filename}' erfolgreich gespeichert!")

check_file_regression(submission_filename)

Datei 'kaltmiete_prediction.csv' erfolgreich gespeichert!
The file is a valid submission for Task 1.


True